# LADA on Google Colab

Setup + run for **LADA: Scalable Label-Specific CLIP Adapter for Continual Learning** (ICML 2025)
on the **X-TAIL** benchmark. Repo: https://github.com/maolinluo/lada

**Before you start:** `Runtime > Change runtime type > Hardware accelerator = GPU` (a T4 is fine).

Two run modes (set in the **Config** cell):
- **`quick`** — 3 small datasets (Caltech101 + EuroSAT + MNIST). Fits on **free Colab**. End-to-end smoke test, ~15-25 min.
- **`full`** — the full 10-dataset benchmark the authors ran. Needs **~80 GB disk** (Colab **Pro** or store on Google Drive).

Constraints baked in: tqdm progress bars during training, GPU memory capped at 16 GB (matches a T4 / RTX 5070 Ti).


## 1. Check the GPU


In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0),
          '| total mem GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))
else:
    raise SystemExit('No GPU! Set Runtime > Change runtime type > GPU, then re-run.')


## 2. Config
Pick the run mode and where to store the dataset.


In [ ]:
MODE = 'full'       # 'quick' (3 small datasets) | 'full' (all 10 datasets, the paper's benchmark)
USE_DRIVE = False   # False = store on Colab disk (faster; Pro has room). True = store on Google Drive (persists; needs ~50GB free there)
SHOTS = 16          # few-shot setting from the paper (16-shot order-I)
OUTPUT_DIR = 'TAIL_colab'
print('MODE =', MODE, '| USE_DRIVE =', USE_DRIVE, '| SHOTS =', SHOTS)


## 3. Clone the repo and install dependencies
Colab already ships a CUDA-enabled PyTorch, so we only install the remaining requirements.


In [ ]:
%cd /content
![ -d lada ] || git clone https://github.com/maolinluo/lada.git
%cd /content/lada
!pip -q install -r requirements.txt modelscope
print('done')


## 4. Patch the code (tqdm progress bar)
Adds a tqdm bar to the training loop (upstream only prints a line every 10 batches).
**No GPU cap** — on Colab the GPU is dedicated, so LADA uses the whole card. Idempotent — safe to re-run.


In [ ]:
from pathlib import Path

# --- trainer.py: add a tqdm progress bar to the training loop ---
t = Path('trainer.py').read_text()

old_loop = ('            num_batches = len(self.train_loader)\n'
            '            for batch_idx, batch in enumerate(self.train_loader):\n'
            '                data_time.update(time.time() - end)')
new_loop = ('            num_batches = len(self.train_loader)\n'
            '            pbar = tqdm(self.train_loader, total=num_batches, ascii=True, leave=False,\n'
            '                        desc=f"Train {cfg.dataset} epoch [{epoch_idx + 1}/{num_epochs}]")\n'
            '            for batch_idx, batch in enumerate(pbar):\n'
            '                data_time.update(time.time() - end)')
if new_loop not in t:
    assert old_loop in t, 'training loop not found (upstream changed?)'
    t = t.replace(old_loop, new_loop)

old_pf = ('                batch_time.update(time.time() - end)\n\n'
          '                meet_freq = (batch_idx + 1) % cfg.print_freq == 0')
new_pf = ('                batch_time.update(time.time() - end)\n\n'
          '                pbar.set_postfix(loss=f"{loss_meter.avg:.4f}",\n'
          '                                 acc=f"{acc_meter.avg:.2f}",\n'
          '                                 lr=f"{current_lr:.2e}")\n\n'
          '                meet_freq = (batch_idx + 1) % cfg.print_freq == 0')
if 'set_postfix' not in t:
    assert old_pf in t
    t = t.replace(old_pf, new_pf)

t = t.replace('                    print(" ".join(info))',
              '                    tqdm.write(" ".join(info))')
Path('trainer.py').write_text(t)
print('patched trainer.py (tqdm only). No GPU cap -> LADA uses the full Colab GPU.')


## 5. Download the X-TAIL dataset
Datasets come from ModelScope. Each zip is downloaded, extracted, then **deleted to save disk**.
`quick` mode pulls only 3 small datasets; `full` pulls all 10 (Sun397 alone is ~36 GB).


In [ ]:
import os, subprocess

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = '/content/drive/MyDrive/lada_data/X-TAIL'
else:
    DATASET_ROOT = '/content/datasets/X-TAIL'
os.makedirs(DATASET_ROOT, exist_ok=True)
print('DATASET_ROOT =', DATASET_ROOT)

QUICK = ['Caltech101.zip', 'EuroSAT.zip', 'MNIST.zip']
FULL  = ['Aircraft.zip', 'Caltech101.zip', 'DTD.zip', 'EuroSAT.zip', 'Flowers.zip',
         'Food.zip', 'MNIST.zip', 'Pets.zip', 'StanfordCars.zip', 'Sun397.zip']
zips = QUICK if MODE == 'quick' else FULL

for z in zips:
    out_dir = os.path.join(DATASET_ROOT, z[:-4])  # zip extracts to a dir of the same name
    if os.path.isdir(out_dir):
        print(f'[skip] {z} (already extracted)')
        continue
    print(f'[download] {z} ...')
    subprocess.run(['modelscope', 'download', '--dataset', 'ForestLuo/X-TAIL',
                    '--local_dir', DATASET_ROOT, z], check=True)
    print(f'[unzip] {z} ...')
    subprocess.run(['unzip', '-q', '-o', os.path.join(DATASET_ROOT, z), '-d', DATASET_ROOT], check=True)
    os.remove(os.path.join(DATASET_ROOT, z))  # free disk
    print(f'[done] {z}')

print('\nExtracted datasets:', sorted(d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))))
!df -h /content | tail -1


## 6. Point the config at the dataset
Writes `dataset_sequence` (3 datasets for quick, all 10 for full) and the dataset `root` into the config.


In [ ]:
seq_quick = ['caltech101', 'eurosat', 'mnist']
seq_full  = ['aircraft', 'caltech101', 'dtd', 'eurosat', 'flowers',
             'food101', 'mnist', 'oxford_pets', 'stanford_cars', 'sun397']
seq = seq_quick if MODE == 'quick' else seq_full

with open('configs/data/TAIL.yaml', 'w') as f:
    f.write('dataset_sequence: %s\n' % seq)
    f.write('root: "%s"\n' % DATASET_ROOT)

print(open('configs/data/TAIL.yaml').read())


## 7. Train + evaluate
Runs each task sequentially (continual learning), then aggregates the final accuracy matrix.
You'll see a **tqdm bar per epoch** with live loss/acc. `quick` is fast; `full` takes a few hours.


In [ ]:
import os, subprocess

NUM_WORKERS = os.cpu_count() or 8   # use all available CPU cores for data loading
print('CPU cores / num_workers =', NUM_WORKERS)

# (dataset, num_epochs, continue_train_first) -- epochs match the authors' run_TAIL_16shot.sh
runs_quick = [('caltech101', 10, True), ('eurosat', 100, False), ('mnist', 200, False)]
runs_full  = [('aircraft', 40, True), ('caltech101', 10, False), ('dtd', 30, False),
              ('eurosat', 100, False), ('flowers', 30, False), ('food101', 5, False),
              ('mnist', 200, False), ('oxford_pets', 10, False), ('stanford_cars', 30, False),
              ('sun397', 10, False)]
runs = runs_quick if MODE == 'quick' else runs_full

for ds, ep, first in runs:
    cmd = ['python', 'main.py', '-d', 'TAIL', '-m', 'clip_vit_b16',
           'num_shots', str(SHOTS), 'dataset', ds, 'num_epochs', str(ep),
           'num_workers', str(NUM_WORKERS), 'output_dir', OUTPUT_DIR]
    if first:
        cmd += ['continue_train_first', 'True']
    print('\n>>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('\n==== FINAL RESULTS ====')
subprocess.run(['python', 'result_process.py', '-d', 'TAIL', '--output_dir', OUTPUT_DIR], check=True)


## Notes & troubleshooting
- **Disk full in `full` mode on free Colab** — expected; Sun397 pushes peak usage past ~80 GB. Use Colab **Pro**, or set `USE_DRIVE = True` (needs a Google One plan with enough space).
- **Session disconnects** — free Colab caps sessions (~12 h) and idle time. `full` is long; keep the tab active or use Pro. With `USE_DRIVE=True` the dataset survives a reconnect, so you only re-run from step 4.
- **`quick` results are not comparable to the paper** — it's a 3-dataset smoke test to confirm the pipeline works. Use `full` for the real benchmark numbers.
- **Order-II** — swap `seq_full` for the order-II sequence and reorder `runs_full` accordingly (see `scripts/run_TAIL_16shot_order2.sh`).
